# 01 — Análise Exploratória de Dados (EDA)

Objetivo: entender o comportamento temporal e estatístico das transações antes da modelagem.

**Suporta:** ULB | IEEE-CIS | Sparkov  
**Altere a variável `DATASET` abaixo para trocar entre datasets.**

In [ ]:
# ── Configuração ─────────────────────────────────────────────
DATASET = "ulb"

PATHS = {
    "ulb":     "../data/processed/ulb_clean.csv",
    "ieee":    "../data/processed/ieee_clean.csv",
    "sparkov": "../data/processed/sparkov_clean.csv",
}

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv(PATHS[DATASET])
print(f"Dataset: {DATASET.upper()}")
print(f"Shape  : {df.shape}")
print(f"Fraudes: {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.3f}%)")
df.head(3)

## 1. Visão Geral do Dataset

In [ ]:
# Tipos, nulos e estatísticas básicas
print("=== Tipos e nulos ===")
info = pd.DataFrame({
    "dtype":    df.dtypes,
    "nulos":    df.isnull().sum(),
    "nulos_%":  (df.isnull().mean() * 100).round(2),
    "únicos":   df.nunique(),
})
print(info[info["nulos"] > 0].to_string() if info["nulos"].sum() > 0 else "Nenhum nulo encontrado.")
print()
print("=== Estatísticas — amount ===")
print(df["amount"].describe().round(2).to_string())

In [ ]:
# Balanço de classes
counts = df["is_fraud"].value_counts()
labels = ["Normal", "Fraude"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(labels, counts.values, color=["steelblue", "crimson"], edgecolor="white", width=0.5)
axes[0].set_title("Contagem de transações por classe")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, v in enumerate(counts.values):
    axes[0].text(i, v + counts.max()*0.01, f"{v:,}", ha="center", fontsize=10)

axes[1].pie(counts.values, labels=labels, autopct="%1.2f%%",
            colors=["steelblue", "crimson"], startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Proporção Normal vs Fraude")

plt.suptitle(f"Desbalanceamento de Classes — {DATASET.upper()}", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/01_class_balance.png", bbox_inches="tight")
plt.show()

## 2. Distribuição de Valores (Amount)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

normal = df[df["is_fraud"] == 0]["amount"]
fraud  = df[df["is_fraud"] == 1]["amount"]

# Histograma geral
axes[0].hist(normal, bins=80, alpha=0.6, color="steelblue", label="Normal")
axes[0].hist(fraud,  bins=80, alpha=0.7, color="crimson",   label="Fraude")
axes[0].set_title("Distribuição de Amount")
axes[0].set_xlabel("Amount")
axes[0].legend()

# Log scale
if "log_amount" in df.columns:
    axes[1].hist(df[df["is_fraud"]==0]["log_amount"], bins=60, alpha=0.6, color="steelblue", label="Normal")
    axes[1].hist(df[df["is_fraud"]==1]["log_amount"], bins=60, alpha=0.7, color="crimson",   label="Fraude")
    axes[1].set_title("Distribuição de log(Amount)")
    axes[1].legend()

# Boxplot
df_box = df[["amount", "is_fraud"]].copy()
df_box["Classe"] = df_box["is_fraud"].map({False: "Normal", True: "Fraude"})
df_box_sample = df_box.groupby("Classe").apply(lambda x: x.sample(min(len(x), 5000), random_state=42))
sns.boxplot(data=df_box_sample, x="Classe", y="amount", palette={"Normal":"steelblue","Fraude":"crimson"}, ax=axes[2])
axes[2].set_title("Boxplot de Amount por Classe")

plt.suptitle(f"Análise de Valores — {DATASET.upper()}", fontsize=13)
plt.tight_layout()
plt.savefig("../reports/figures/02_amount_distribution.png", bbox_inches="tight")
plt.show()

print(f"\nAmount médio — Normal: {normal.mean():.2f} | Fraude: {fraud.mean():.2f}")
print(f"Amount mediana — Normal: {normal.median():.2f} | Fraude: {fraud.median():.2f}")

## 3. Análise Temporal

In [ ]:
if "hour" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Frequência por hora
    hourly = df.groupby(["hour", "is_fraud"]).size().unstack(fill_value=0)
    hourly.columns = ["Normal", "Fraude"]
    hourly.plot(kind="bar", ax=axes[0], color=["steelblue", "crimson"],
                edgecolor="white", width=0.8)
    axes[0].set_title("Transações por Hora do Dia")
    axes[0].set_xlabel("Hora")
    axes[0].tick_params(axis="x", rotation=0)

    # Taxa de fraude por hora
    fraud_rate = df.groupby("hour")["is_fraud"].mean() * 100
    axes[1].plot(fraud_rate.index, fraud_rate.values, color="crimson", linewidth=2, marker="o", markersize=4)
    axes[1].axhspan(0, 5, alpha=0.08, color="orange", label="Madrugada (0h–5h)")
    axes[1].set_title("Taxa de Fraude (%) por Hora")
    axes[1].set_xlabel("Hora")
    axes[1].set_ylabel("%")
    axes[1].legend()

    plt.suptitle(f"Padrão Temporal — {DATASET.upper()}", fontsize=13)
    plt.tight_layout()
    plt.savefig("../reports/figures/03_hourly_pattern.png", bbox_inches="tight")
    plt.show()
else:
    print("Coluna 'hour' não encontrada. Execute preprocess.py primeiro.")

## 4. Heatmap — Hora × Dia da Semana

In [ ]:
if "hour" in df.columns and "day_of_week" in df.columns:
    pivot = df.pivot_table(
        values="is_fraud",
        index="day_of_week",
        columns="hour",
        aggfunc="mean"
    ) * 100

    day_labels = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]

    plt.figure(figsize=(16, 4))
    sns.heatmap(
        pivot, cmap="YlOrRd", linewidths=0.3,
        yticklabels=[day_labels[i] for i in pivot.index if i < len(day_labels)],
        cbar_kws={"label": "Taxa de Fraude (%)"},
        fmt=".1f"
    )
    plt.title(f"Heatmap: Taxa de Fraude por Hora × Dia da Semana — {DATASET.upper()}")
    plt.xlabel("Hora do Dia")
    plt.ylabel("Dia da Semana")
    plt.tight_layout()
    plt.savefig("../reports/figures/04_heatmap_fraud.png", bbox_inches="tight")
    plt.show()
else:
    print("Colunas 'hour' e 'day_of_week' necessárias. Disponível para IEEE-CIS e Sparkov.")

## 5. Outliers — Z-Score e IQR

In [ ]:
from scipy import stats

# Z-Score
z_scores = np.abs(stats.zscore(df["amount"].dropna()))
n_outliers_z = (z_scores > 3).sum()

# IQR
Q1, Q3 = df["amount"].quantile(0.25), df["amount"].quantile(0.75)
IQR = Q3 - Q1
n_outliers_iqr = ((df["amount"] < Q1 - 1.5*IQR) | (df["amount"] > Q3 + 1.5*IQR)).sum()

print(f"Outliers por Z-Score (|Z|>3) : {n_outliers_z:,}")
print(f"Outliers por IQR             : {n_outliers_iqr:,}")
print(f"IQR limite inferior          : {Q1 - 1.5*IQR:.2f}")
print(f"IQR limite superior          : {Q3 + 1.5*IQR:.2f}")

# Qual % dos outliers são fraudes?
outlier_mask = z_scores > 3
if len(outlier_mask) == len(df):
    fraud_in_outliers = df[outlier_mask]["is_fraud"].mean() * 100
    print(f"\n% de fraudes entre outliers Z-Score: {fraud_in_outliers:.1f}%")
    print(f"% de fraudes no geral              : {df['is_fraud'].mean()*100:.3f}%")

## 6. Features Específicas por Dataset

In [ ]:
# Sparkov: distribuição geográfica
if DATASET == "sparkov" and "geo_distance" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    df[df["is_fraud"]==0]["geo_distance"].hist(bins=60, ax=axes[0], color="steelblue", alpha=0.7, label="Normal")
    df[df["is_fraud"]==1]["geo_distance"].hist(bins=60, ax=axes[0], color="crimson",   alpha=0.7, label="Fraude")
    axes[0].set_title("Distância Geográfica Cliente ↔ Estabelecimento")
    axes[0].legend()

    top_cat = df.groupby("category")["is_fraud"].mean().sort_values(ascending=False).head(10) * 100
    top_cat.plot(kind="barh", ax=axes[1], color="crimson", edgecolor="white")
    axes[1].set_title("Taxa de Fraude por Categoria (%)")
    axes[1].set_xlabel("%")
    plt.tight_layout()
    plt.savefig("../reports/figures/05_sparkov_geo_category.png", bbox_inches="tight")
    plt.show()

# IEEE-CIS: campos ausentes
elif DATASET == "ieee" and "has_email" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col, label in zip(axes, ["has_email", "has_device"], ["Tem E-mail", "Tem Dispositivo"]):
        rates = df.groupby(col)["is_fraud"].mean() * 100
        rates.index = ["Não", "Sim"]
        rates.plot(kind="bar", ax=ax, color=["crimson", "steelblue"], edgecolor="white", rot=0)
        ax.set_title(f"Taxa de Fraude × {label}")
        ax.set_ylabel("%")
    plt.suptitle("IEEE-CIS: Campos Ausentes como Sinal de Fraude", fontsize=12)
    plt.tight_layout()
    plt.savefig("../reports/figures/05_ieee_missing_signals.png", bbox_inches="tight")
    plt.show()

# ULB: correlação das features V com fraude
elif DATASET == "ulb":
    v_cols = [c for c in df.columns if c.startswith("V")]
    if v_cols:
        corr = df[v_cols + ["is_fraud"]].corr()["is_fraud"].drop("is_fraud").abs().sort_values(ascending=False)
        plt.figure(figsize=(14, 4))
        corr.head(15).plot(kind="bar", color="steelblue", edgecolor="white")
        plt.title("ULB: Top 15 Features V — Correlação com Fraude (|r|)")
        plt.ylabel("|Correlação|")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig("../reports/figures/05_ulb_feature_correlation.png", bbox_inches="tight")
        plt.show()

## 7. Testes de Hipótese

In [ ]:
from scipy.stats import ttest_ind, chi2_contingency

fraud_amt  = df[df["is_fraud"] == 1]["amount"]
normal_amt = df[df["is_fraud"] == 0]["amount"]

t_stat, p_value = ttest_ind(fraud_amt, normal_amt, equal_var=False)
print("=== Teste t — Amount: Fraude vs Normal ===")
print(f"  Média fraude : {fraud_amt.mean():.4f}")
print(f"  Média normal : {normal_amt.mean():.4f}")
print(f"  t-statistic  : {t_stat:.4f}")
print(f"  p-value      : {p_value:.2e}")
print(f"  Conclusão    : {'Diferença significativa ✓' if p_value < 0.05 else 'Sem diferença significativa ✗'}")

if "hour" in df.columns:
    print("\n=== Qui-quadrado — Hora × Fraude ===")
    df["hour_group"] = pd.cut(df["hour"], bins=[0,6,12,18,24], labels=["Madrugada","Manhã","Tarde","Noite"], right=False)
    contingency = pd.crosstab(df["hour_group"], df["is_fraud"])
    chi2, p, dof, _ = chi2_contingency(contingency)
    print(f"  Chi2     : {chi2:.2f}")
    print(f"  p-value  : {p:.2e}")
    print(f"  Conclusão: {'Horário influencia fraude ✓' if p < 0.05 else 'Horário não influencia ✗'}")

## 8. Resumo dos Insights

In [ ]:
print(f"{'='*55}")
print(f"  RESUMO EDA — {DATASET.upper()}")
print(f"{'='*55}")
print(f"  Total transações : {len(df):,}")
print(f"  Fraudes          : {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.3f}%)")
print(f"  Amount médio     : {df['amount'].mean():.2f}")
print(f"  Amount máximo    : {df['amount'].max():.2f}")

if "is_madrugada" in df.columns:
    mad_rate = df[df["is_madrugada"]==1]["is_fraud"].mean() * 100
    geral_rate = df["is_fraud"].mean() * 100
    print(f"  Taxa fraude madrugada : {mad_rate:.2f}% vs {geral_rate:.3f}% geral")

if "is_high_value" in df.columns:
    hv_rate = df[df["is_high_value"]==1]["is_fraud"].mean() * 100
    print(f"  Taxa fraude alto valor: {hv_rate:.2f}%")

print(f"{'='*55}")
print("Próximo passo: notebooks/02_feature_engineering.ipynb")